---
**题名：** 上下文并行（Context Parallelism, CP）

**类目：** context-parallelism

**难度：** 上

**预计用时：** 三刻钟

---

## 总论

上下文并行者，令多器共分长序列之注意力计算也。本篇逐步阐明其理：

一、长序列之注意力何以耗尽存储
二、序列之切分于多器
三、环形注意力之术——旋转KV块
四、在线softmax修正（数值精确无误）
五、完整模拟之实践
六、Megatron-LM之实现

### 先修之识
- 缩放点积注意力（Q、K、V）

> **Q、K、V何谓？** 注意力之法，将输入投射为三阵：**Query**（吾所求者何？）、**Key**（吾所含者何？）、**Value**（吾所载之信息何？）。以Q匹K得注意力之分数，复以此分数加权于V。
- [00-gpu-communication](../00-gpu-communication/) — P2P与环形通信之法
- [04-sequence-parallelism](../04-sequence-parallelism/) — 序列并行之基

## 理法

### 其一：注意力与存储之困

自微始。此乃 **八词元**、**head_dim=4** 之注意力也：

> **head_dim何谓？** 每注意力头所操之向量维度也。小于模型之全隐藏维度——如四千零九十六维之模型有三十二头，则head_dim为一百二十八。

In [ ]:
import torch
import numpy as np
from mp_tutorial.viz import draw_attention_heatmap

torch.manual_seed(0)
S, D = 8, 4  # 8个token，head_dim = 4
Q = torch.randn(S, D)
K = torch.randn(S, D)
V = torch.randn(S, D)

# 计算注意力分数
scale = D ** -0.5
scores = (Q @ K.T) * scale
weights = torch.softmax(scores, dim=-1)

# 可视化：注意力权重矩阵是 S x S
token_labels = [f"t{i}" for i in range(S)]
draw_attention_heatmap(weights, title=f"Attention Weights ({S}x{S})", token_labels=token_labels)

此八行八列之阵尚小。然注意力之存储随 **S x S** 而增——序列倍之，则存储四倍焉：

| 序列之长 | 分数矩阵之大 | 存储 (FP16) |
|---|---|---|
| 4,096 | 一千六百万 | 32 MB |
| 32,768 | 十亿 | 2 GB |
| 131,072 | 百七十亿 | 32 GB |

> **FP16何谓？** 半精度浮点数也——每数用二字节而非四字节（FP32）。训练大模型之常法，以省存储。

及十二万八千词元时，仅分数矩阵已逎多数GPU之容。**上下文并行可解此困。**

### 其二：分序列于多器

其法：予每器一**段**序列。二器八词元，每器得四词元。

> **Chunk（块）何谓？** 序列之一段连续切片也。有八词元二器，则词元零至三为块零，词元四至七为块一。

In [ ]:
from mp_tutorial.viz import draw_tensor_blocks

num_gpus = 2
chunk_size = S // num_gpus

# 切分 Q, K, V
q_chunks = list(Q.split(chunk_size, dim=0))
k_chunks = list(K.split(chunk_size, dim=0))
v_chunks = list(V.split(chunk_size, dim=0))

draw_tensor_blocks(q_chunks, title=f"Q split across {num_gpus} GPUs ({chunk_size} tokens each)")
draw_tensor_blocks(k_chunks, title=f"K split across {num_gpus} GPUs")

In [ ]:
from mp_tutorial.viz import draw_context_partition

# 一个真实句子切分到2个GPU — each GPU gets half the context
tokens = ["The", "cat", "sat", "on", "the", "mat", "and", "slept"]
draw_context_partition(tokens, num_gpus=2, q_chunks=q_chunks,
                        title="Context Parallelism: 8 tokens split across 2 GPUs")

In [ ]:
# 同一句子，现在用4个GPU — each gets just 2 tokens
q4 = list(Q.split(2, dim=0))
draw_context_partition(tokens, num_gpus=4, q_chunks=q4,
                        title="4-way CP: each GPU holds only 2 tokens of context")

每器各有其**本地Q块**。然欲正确计算注意力，每Q须见**全部**K与V——非止本地之块。

完整注意力矩阵有四tile。观之：

In [ ]:
# 显示带有块边界的完整注意力矩阵
draw_attention_heatmap(
    weights,
    title="Full Attention — divided into tiles by GPU chunk",
    chunk_boundaries=[chunk_size],
    token_labels=token_labels
)

红线者，块之边界也。每器须算**其行之tile**（每KV块对应一tile）。

> **Tile何谓？** 完整注意力矩阵之一子块也。二器时，S×S之矩阵分为二乘二之tile格，每tile大小为(S/2)×(S/2)。

- GPU 0须：tile(Q0, K0) 与 tile(Q0, K1)
- GPU 1须：tile(Q1, K0) 与 tile(Q1, K1)

然每器初仅有己之KV块。如何取他者之块乎？

### 其三：环形注意力——旋转KV块

不必聚合全部KV于每器（费甚！），但令**KV块于环上传递**可也：

> **P2P（点对点）何谓？** 器与器之直接通信也——GPU 0径将数据送于GPU 1，不经中枢。较之广播于众，速更捷也。

In [ ]:
from mp_tutorial.viz import draw_ring_step_dataflow

# 第0步：每个GPU使用自己的本地KV
local_k = list(k_chunks)  # GPU 0 has K0, GPU 1 has K1
local_v = list(v_chunks)

step0_scores = []
step0_outputs = []
for gpu in range(num_gpus):
    s = (q_chunks[gpu] @ local_k[gpu].T) * scale
    w = torch.softmax(s, dim=-1)
    step0_scores.append(s)
    step0_outputs.append(w @ local_v[gpu])

draw_ring_step_dataflow(
    q_chunks, local_k, local_v, step0_scores, step0_outputs,
    step=0, num_gpus=num_gpus,
    title="Ring Step 0: Each GPU attends to its LOCAL KV"
)

In [ ]:
from mp_tutorial.distributed import simulate_p2p_kv_exchange

# KV旋转一步: GPU 0 gets K1, GPU 1 gets K0
local_k = simulate_p2p_kv_exchange(local_k)
local_v = simulate_p2p_kv_exchange(local_v)

step1_scores = []
step1_outputs = []
for gpu in range(num_gpus):
    s = (q_chunks[gpu] @ local_k[gpu].T) * scale
    w = torch.softmax(s, dim=-1)
    step1_scores.append(s)
    step1_outputs.append(w @ local_v[gpu])

draw_ring_step_dataflow(
    q_chunks, local_k, local_v, step1_scores, step1_outputs,
    step=1, num_gpus=num_gpus,
    title="Ring Step 1: KV blocks rotated — each GPU sees the OTHER chunk"
)

二步（等于器之数）既毕，每器已遍历全部KV块矣！无一器曾持全序列。

然不可径取二部分输出之均值。须用**在线softmax修正**。

### 其四：在线Softmax修正

其难：对全部key维做softmax，须一览所有分数。然吾等逐tile而算之。

其解：维持滚动之max与exp之和，新tile至则**重标**累积之输出。观一具体之例：

> **在线算法何谓？** 逐块处理数据而维持滚动之结果，不必一举得全部数据之算法也。此处吾等逐tile累积注意力之结果，无须存全S×S之矩阵。

In [ ]:
# 逐步演示GPU 0的在线softmax修正
q0 = q_chunks[0]  # (4, 4)
k0_orig = k_chunks[0]  # K from chunk 0
k1_orig = k_chunks[1]  # K from chunk 1

print("=== GPU 0: Online Softmax Correction ===\n")

# --- Tile 0: Q0 @ K0 ---
scores_t0 = (q0 @ k0_orig.T) * scale
m0 = scores_t0.max(dim=-1, keepdim=True).values
exp0 = torch.exp(scores_t0 - m0)
l0 = exp0.sum(dim=-1, keepdim=True)
o0 = exp0 @ v_chunks[0]

print("Tile 0 (Q0 x K0):")
print(f"  scores shape: {scores_t0.shape}")
print(f"  row max (m):  {m0.squeeze().tolist()}")
print(f"  sum(exp):     {l0.squeeze().tolist()}")

# --- Tile 1: Q0 @ K1 ---
scores_t1 = (q0 @ k1_orig.T) * scale
m1 = scores_t1.max(dim=-1, keepdim=True).values
exp1 = torch.exp(scores_t1 - m1)
l1 = exp1.sum(dim=-1, keepdim=True)
o1 = exp1 @ v_chunks[1]

print(f"\nTile 1 (Q0 x K1):")
print(f"  row max (m):  {m1.squeeze().tolist()}")
print(f"  sum(exp):     {l1.squeeze().tolist()}")

# --- Merge: online correction ---
m_new = torch.maximum(m0, m1)
corr0 = torch.exp(m0 - m_new)  # rescale factor for tile 0
corr1 = torch.exp(m1 - m_new)  # rescale factor for tile 1
l_merged = corr0 * l0 + corr1 * l1
o_merged = (corr0 * o0 + corr1 * o1) / l_merged

print(f"\nMerged:")
print(f"  new max:        {m_new.squeeze().tolist()}")
print(f"  correction t0:  {corr0.squeeze().tolist()}")
print(f"  correction t1:  {corr1.squeeze().tolist()}")

# Verify against standard attention for GPU 0's rows
ref = (torch.softmax((q0 @ torch.cat([k0_orig, k1_orig]).T) * scale, dim=-1)
       @ torch.cat([v_chunks[0], v_chunks[1]]))
print(f"\nMax error vs standard attention: {(o_merged - ref).abs().max():.1e}")

修正因子 `exp(m_old - m_new)` 者，于新tile有更大max时，重标先前之累积值也。此与FlashAttention所用之术全同——环形注意力但推广之于分布式耳。

**每tile之更新法则：**

$$m_{\text{new}} = \max(m, m_k), \quad \ell_{\text{new}} = e^{m - m_{\text{new}}} \ell + e^{m_k - m_{\text{new}}} \ell_k, \quad O_{\text{new}} = e^{m - m_{\text{new}}} O + e^{m_k - m_{\text{new}}} O_k$$

## 图解

### 环形注意力之全程（四器）

今扩至四器，观全环之流转：

In [ ]:
from mp_tutorial.viz import draw_ring_attention_steps

draw_ring_attention_steps(num_gpus=4, num_steps=4, title="Ring Attention: 4 GPUs, 4 Steps")

### 存储之消长

其利：有N器时，每器每时仅存一 (S/N) x (S/N) 之tile。

In [ ]:
from mp_tutorial.viz import draw_cp_memory_scaling

draw_cp_memory_scaling(seq_lengths=[4096, 32768, 131072], max_gpus=8)

## 于大语言模型之用

| 并行之维 | 所切分者 | 通常规模 |
|---|---|---|
| **DP** | 批次 | 64-1024倍 |
| **TP** | 权重矩阵/注意力头 | 2-8倍（节点内） |
| **SP** | LayerNorm、Dropout之激活值 | 同TP |
| **CP** | 注意力Q/K/V沿序列维 | 2-16倍 |
| **PP** | 层 | 4-16倍 |


> **NVLink何谓？** 同机（节点）内GPU间之高速直连也。远快于走网络，故TP通常仅限于NVLink所连之GPU。

CP与TP/SP正交——TP切**头**，CP切**序列**。二者可合用之。

**用者：** Megatron-LM（`--context-parallel-size`），Llama 3.1（128K上下文训练），DeepSpeed Ulysses（all-to-all之变体）。

## 实操

### 环形注意力之完整模拟

运行全算法，验其与标准注意力之一致：

In [ ]:
from mp_tutorial.distributed import check_gpu_env
check_gpu_env()

In [ ]:
from mp_tutorial.distributed import simulate_ring_attention

torch.manual_seed(42)
S, D, N = 32, 16, 4
Q = torch.randn(S, D)
K = torch.randn(S, D)
V = torch.randn(S, D)

# 环形注意力
ring_out = simulate_ring_attention(Q, K, V, num_gpus=N, verbose=True)

# 标准注意力
ref = torch.softmax((Q @ K.T) * (D ** -0.5), dim=-1) @ V

print(f"\nMax error: {(ring_out - ref).abs().max():.1e}")
print(f"Match: {torch.allclose(ring_out, ref, atol=1e-5)}")

### 扩展之验

器愈多则结果不变，唯切分之法异耳：

In [ ]:
from mp_tutorial.formatting import comparison_table

torch.manual_seed(123)
S, D = 64, 32
Q = torch.randn(S, D)
K = torch.randn(S, D)
V = torch.randn(S, D)
ref = torch.softmax((Q @ K.T) * (D ** -0.5), dim=-1) @ V

rows = []
for n_gpus in [2, 4, 8]:
    out = simulate_ring_attention(Q, K, V, num_gpus=n_gpus)
    err = (out - ref).abs().max().item()
    rows.append([str(n_gpus), str(S // n_gpus), f"{err:.1e}",
                 "yes" if torch.allclose(out, ref, atol=1e-5) else "no"])

comparison_table(
    ["GPUs", "Chunk size", "Max error", "Match"],
    rows,
    title="Ring Attention correctness across GPU counts"
)

> **须GPU** — 于多卡 GPU 机器运行此 cell（推荐四卡以上）。

In [ ]:
# [GPU-REQUIRED]
# 真实分布式环形注意力 with torch.distributed
# 启动方式：torchrun --nproc_per_node=4 this_script.py

import torch
import torch.distributed as dist

def ring_attention_distributed(q_local, k_local, v_local, group=None):
    """使用真实P2P通信的环形注意力。"""
    rank = dist.get_rank(group)
    world_size = dist.get_world_size(group)
    chunk_size, head_dim = q_local.shape
    scale = head_dim ** -0.5

    m = torch.full((chunk_size, 1), float("-inf"), device=q_local.device)
    l = torch.zeros(chunk_size, 1, device=q_local.device)
    o = torch.zeros_like(q_local)

    k_recv = torch.empty_like(k_local)
    v_recv = torch.empty_like(v_local)
    k_curr, v_curr = k_local, v_local

    for step in range(world_size):
        # 在计算前启动异步P2P（通信与计算重叠）
        if step < world_size - 1:
            send_rank = (rank + 1) % world_size
            recv_rank = (rank - 1) % world_size
            send_k = dist.isend(k_curr, dst=send_rank, group=group)
            send_v = dist.isend(v_curr, dst=send_rank, group=group)
            recv_k = dist.irecv(k_recv, src=recv_rank, group=group)
            recv_v = dist.irecv(v_recv, src=recv_rank, group=group)

        # 计算注意力tile
        scores = (q_local @ k_curr.T) * scale
        tile_max = scores.max(dim=-1, keepdim=True).values
        tile_exp = torch.exp(scores - tile_max)
        tile_sum = tile_exp.sum(dim=-1, keepdim=True)
        tile_out = tile_exp @ v_curr

        # 在线softmax修正
        m_new = torch.maximum(m, tile_max)
        l = torch.exp(m - m_new) * l + torch.exp(tile_max - m_new) * tile_sum
        o = torch.exp(m - m_new) * o + torch.exp(tile_max - m_new) * tile_out
        m = m_new

        if step < world_size - 1:
            send_k.wait(); send_v.wait()
            recv_k.wait(); recv_v.wait()
            k_curr, v_curr = k_recv.clone(), v_recv.clone()

    return o / l

# dist.init_process_group("nccl")
# rank = dist.get_rank()
# ... 切分Q, K, V并调用ring_attention_distributed() ...

## Megatron之参

Megatron-LM于 `megatron/core/transformer/dot_product_attention.py` 中实现CP。

In [ ]:
from mp_tutorial.formatting import code_reference

code_reference(
    code="""# Megatron环形注意力的关键设计选择：
#
# 1. 通信-计算重叠
#    - KV的isend/irecv在计算前启动
#    - GPU计算注意力tile的同时网络传输下一个KV块
#
# 2. 因果掩码优化
#    - 对于自回归模型，GPU i跳过来自
#      位置 > i 的KV块（未来token无法注意到）
#    - 对因果模型可节省约50%的环形步骤
#
# 3. CP + TP 集成
#    - CP组：切分序列维度
#    - TP组：切分注意力头
#    - 正交 — 两者在同一前向传播中同时活动
#    - 总GPU数 = DP x TP x PP x CP""",
    source="Megatron-LM",
    filepath="megatron/core/transformer/dot_product_attention.py"
)

## 总括与延读

**要旨：**
- CP分注意力于多器：每器存储自O(S^2)降至O(S^2/N)
- 环形注意力于环上旋转KV块——每器同时仅持一块
- 在线softmax修正令分块注意力数值精确（与FlashAttention同术）
- CP与TP/SP正交——TP切头，CP切序列，组合无碍
- 通信与计算可重叠；因果掩码可省约半数环形步骤

> **因果掩码何谓？** 自回归（自左而右）之模型中，词元i仅可注意词元零至i，不可窥未来之词元。因果掩码者，屏蔽序列中后方位置之注意力以行此约束也。

**延读：**
- [Ring Attention with Blockwise Transformers](https://arxiv.org/abs/2310.01889) — Liu et al., 2023
- [FlashAttention-2](https://arxiv.org/abs/2307.08691) — Dao, 2023（在线softmax修正之源）
- [Megatron-LM](https://github.com/NVIDIA/Megatron-LM) — NVIDIA分布式训练之框架
- [DeepSpeed Ulysses](https://arxiv.org/abs/2309.14509) — all-to-all CP之变体